# Day 6 — Advanced Analytics & Risk Metrics
**Bluestock Fintech Capstone | Mutual Fund Analytics Platform**

This notebook covers advanced risk and behavioural analytics:
- Historical VaR and CVaR (95%) for all 40 funds
- Rolling 90-day Sharpe ratio over time
- Investor cohort analysis by first transaction year
- SIP continuity analysis — flagging at-risk investors
- Fund recommender by risk appetite
- Sector HHI concentration per equity fund


## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # remove this line when running in Jupyter locally
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')



ROOT    = Path().resolve().parent
RAW     = ROOT / 'data' / 'raw'
PROC    = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi']        = 120
plt.rcParams['font.family']       = 'DejaVu Sans'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print("Setup done.")


## 2. Load Data

In [ ]:
nav_df   = pd.read_csv(PROC / '02_nav_history_clean.csv')
nav_df['date'] = pd.to_datetime(nav_df['date'])

tx_df    = pd.read_csv(PROC / '08_investor_transactions_clean.csv')
tx_df['transaction_date'] = pd.to_datetime(tx_df['transaction_date'])

perf_df  = pd.read_csv(PROC / '07_scheme_performance_clean.csv')
fund_df  = pd.read_csv(PROC / '01_fund_master_clean.csv')
port_df  = pd.read_csv(PROC / '09_portfolio_holdings_clean.csv')

fund_names = fund_df.set_index('amfi_code')['scheme_name'].to_dict()

# nav_pct_change is already computed in the cleaned dataset
# divide by 100 to get decimal returns (it's stored as percentage)
nav_df['daily_return'] = nav_df['nav_pct_change'] / 100

print(f"NAV rows         : {len(nav_df):,} | schemes: {nav_df['amfi_code'].nunique()}")
print(f"Transaction rows : {len(tx_df):,}")
print(f"Portfolio rows   : {len(port_df):,}")


## 3. Historical VaR and CVaR (95%)

**VaR (Value at Risk)** at 95% confidence = the 5th percentile of daily returns.
It answers: *on a bad day (1 in 20), what is the worst loss you can expect?*

**CVaR (Conditional VaR / Expected Shortfall)** = average of all returns *below* the VaR threshold.
It answers: *if things go worse than VaR, how bad does it actually get on average?*

Both are expressed as % of portfolio value.


In [ ]:
var_records = []

returns_clean = nav_df.dropna(subset=['daily_return'])

for code, group in returns_clean.groupby('amfi_code'):
    r = group['daily_return'].dropna()
    if len(r) < 50:
        continue

    var_95  = np.percentile(r, 5)                    # 5th percentile
    cvar_95 = r[r <= var_95].mean()                  # mean of tail losses
    mean_r  = r.mean() * 252                         # annualised return
    std_r   = r.std() * np.sqrt(252)                 # annualised vol

    var_records.append({
        'amfi_code'       : code,
        'scheme_name'     : fund_names.get(code, str(code)),
        'var_95_pct'      : round(var_95 * 100, 4),
        'cvar_95_pct'     : round(cvar_95 * 100, 4),
        'ann_return_pct'  : round(mean_r * 100, 2),
        'ann_vol_pct'     : round(std_r * 100, 2),
        'n_days'          : len(r),
    })

var_df = pd.DataFrame(var_records).sort_values('var_95_pct').reset_index(drop=True)
var_df.to_csv(REPORTS / 'var_cvar_report.csv', index=False)

print("var_cvar_report.csv saved.")
print()
print("10 funds with worst VaR (highest daily loss risk):")
print(var_df[['scheme_name','var_95_pct','cvar_95_pct','ann_return_pct']].head(10).to_string(index=False))
print()
print(f"VaR range   : {var_df['var_95_pct'].min():.3f}% to {var_df['var_95_pct'].max():.3f}%")
print(f"CVaR range  : {var_df['cvar_95_pct'].min():.3f}% to {var_df['cvar_95_pct'].max():.3f}%")


In [ ]:
# VaR vs CVaR bar chart for all 40 funds
fig, ax = plt.subplots(figsize=(14, 7))

x      = np.arange(len(var_df))
width  = 0.4
names  = [n[:30] for n in var_df['scheme_name']]

bars1 = ax.bar(x - width/2, var_df['var_95_pct'],  width, label='VaR 95%',  color='#1565C0', alpha=0.85)
bars2 = ax.bar(x + width/2, var_df['cvar_95_pct'], width, label='CVaR 95%', color='#C62828', alpha=0.75)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=75, ha='right', fontsize=7)
ax.set_ylabel('Daily Loss (%)', fontsize=11)
ax.set_title('VaR and CVaR (95%) — All 40 Funds\n(More negative = higher risk)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(var_df['var_95_pct'].mean(), color='#1565C0', linestyle='--',
           linewidth=1, alpha=0.6, label='Avg VaR')
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(REPORTS / 'var_cvar_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: var_cvar_chart.png")


## 4. Rolling 90-Day Sharpe Ratio

Shows how the risk-adjusted performance of a fund changes over time.
A falling rolling Sharpe means the fund is getting riskier or returns are dropping.
A rising Sharpe means improving risk-adjusted performance.

`Rolling Sharpe = rolling_mean(return - Rf_daily) / rolling_std(return) × √252`


In [ ]:
RF_DAILY = 0.065 / 252

# pick 5 funds — one from each major category for variety
top5_codes = perf_df.groupby('category').apply(
    lambda g: g.nlargest(1, 'sharpe_ratio')
).reset_index(drop=True)['amfi_code'].head(5).tolist()

fig, ax = plt.subplots(figsize=(14, 6))
colors  = ['#1565C0','#2E7D32','#E65100','#6A1B9A','#00796B']

for i, code in enumerate(top5_codes):
    group = returns_clean[returns_clean['amfi_code'] == code].sort_values('date').copy()
    r     = group.set_index('date')['daily_return']

    rolling_mean = (r - RF_DAILY).rolling(90).mean()
    rolling_std  = r.rolling(90).std()
    rolling_sharpe = (rolling_mean / rolling_std * np.sqrt(252)).dropna()

    name = fund_names.get(code, str(code))[:35]
    ax.plot(rolling_sharpe.index, rolling_sharpe.values,
            color=colors[i], linewidth=1.8, label=name, alpha=0.9)

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Sharpe = 1.0 (good)')
ax.axhline(0.0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
ax.set_title('Rolling 90-Day Sharpe Ratio — Top 5 Funds by Category',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Rolling Sharpe Ratio', fontsize=11)
ax.legend(fontsize=9, loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(REPORTS / 'rolling_sharpe_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: rolling_sharpe_chart.png")
print("Funds plotted:")
for code in top5_codes:
    print(f"  {code}: {fund_names.get(code, str(code))[:55]}")


## 5. Investor Cohort Analysis

Grouping investors by the **year of their first transaction**.
This shows whether newer investors behave differently from older ones —
do they invest more? prefer different funds? have higher SIP amounts?


In [ ]:
sip_tx = tx_df[tx_df['transaction_type'] == 'SIP'].copy()

# find first transaction year per investor
first_tx = tx_df.groupby('investor_id')['transaction_date'].min().reset_index()
first_tx['cohort_year'] = first_tx['transaction_date'].dt.year
first_tx = first_tx[['investor_id','cohort_year']]

# merge cohort year back into all transactions
tx_cohort = tx_df.merge(first_tx, on='investor_id', how='left')
sip_cohort = tx_cohort[tx_cohort['transaction_type'] == 'SIP']

# cohort metrics
cohort_stats = sip_cohort.groupby('cohort_year').agg(
    num_investors    = ('investor_id', 'nunique'),
    total_invested   = ('amount_inr',  'sum'),
    avg_sip_amount   = ('amount_inr',  'mean'),
    num_transactions = ('investor_id', 'count'),
).reset_index()

cohort_stats['avg_sip_amount']   = cohort_stats['avg_sip_amount'].round(0).astype(int)
cohort_stats['total_invested_cr'] = (cohort_stats['total_invested'] / 1e7).round(2)

# top fund per cohort
top_fund_per_cohort = (
    sip_cohort.groupby(['cohort_year','amfi_code'])
    .size().reset_index(name='count')
    .sort_values('count', ascending=False)
    .groupby('cohort_year').first().reset_index()
)
top_fund_per_cohort['fund_name'] = top_fund_per_cohort['amfi_code'].map(fund_names).str[:40]

cohort_stats = cohort_stats.merge(
    top_fund_per_cohort[['cohort_year','fund_name']], on='cohort_year', how='left'
)

cohort_stats.to_csv(REPORTS / 'cohort_analysis.csv', index=False)
print("cohort_analysis.csv saved.")
print()
print(cohort_stats[['cohort_year','num_investors','avg_sip_amount',
                     'total_invested_cr','num_transactions','fund_name']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# chart 1 — avg SIP amount by cohort
axes[0].bar(cohort_stats['cohort_year'].astype(str),
            cohort_stats['avg_sip_amount'],
            color=['#1565C0','#1E88E5','#64B5F6'][:len(cohort_stats)],
            width=0.5, edgecolor='white')
axes[0].set_title('Avg SIP Amount by Cohort Year', fontweight='bold')
axes[0].set_ylabel('Avg SIP Amount (₹)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
                 f'₹{bar.get_height():,.0f}', ha='center', fontsize=9)

# chart 2 — total invested by cohort
axes[1].bar(cohort_stats['cohort_year'].astype(str),
            cohort_stats['total_invested_cr'],
            color=['#2E7D32','#43A047','#81C784'][:len(cohort_stats)],
            width=0.5, edgecolor='white')
axes[1].set_title('Total SIP Invested by Cohort (₹ Crore)', fontweight='bold')
axes[1].set_ylabel('₹ Crore')

plt.suptitle('Investor Cohort Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(REPORTS / 'cohort_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: cohort_chart.png")


## 6. SIP Continuity Analysis

A healthy SIP should have consistent monthly transactions — roughly 30 days apart.
Investors with gaps > 35 days may have paused or stopped their SIP.
We flag these as **"at-risk"** since lapsed SIPs mean lost AUM for the AMC.


In [ ]:
sip_only = tx_df[tx_df['transaction_type'] == 'SIP'].sort_values(
    ['investor_id','transaction_date']
).copy()

# only analyse investors with 6+ SIP transactions
sip_count = sip_only.groupby('investor_id').size()
eligible  = sip_count[sip_count >= 6].index
sip_elig  = sip_only[sip_only['investor_id'].isin(eligible)]

continuity_records = []

for inv_id, group in sip_elig.groupby('investor_id'):
    dates = group['transaction_date'].sort_values().reset_index(drop=True)
    gaps  = dates.diff().dt.days.dropna()

    avg_gap    = gaps.mean()
    max_gap    = gaps.max()
    num_tx     = len(dates)
    at_risk    = avg_gap > 35

    continuity_records.append({
        'investor_id' : inv_id,
        'num_sip_tx'  : num_tx,
        'avg_gap_days': round(avg_gap, 1),
        'max_gap_days': int(max_gap),
        'first_sip'   : dates.iloc[0].date(),
        'last_sip'    : dates.iloc[-1].date(),
        'status'      : 'At-Risk' if at_risk else 'Active',
    })

cont_df = pd.DataFrame(continuity_records)
cont_df.to_csv(REPORTS / 'sip_continuity.csv', index=False)

total     = len(cont_df)
at_risk_n = (cont_df['status'] == 'At-Risk').sum()
active_n  = total - at_risk_n

print("sip_continuity.csv saved.")
print()
print(f"Investors analysed (6+ SIPs) : {total:,}")
print(f"Active (avg gap <= 35 days)  : {active_n:,}  ({active_n/total*100:.1f}%)")
print(f"At-Risk (avg gap > 35 days)  : {at_risk_n:,}  ({at_risk_n/total*100:.1f}%)")
print()
print("Gap distribution:")
print(cont_df['avg_gap_days'].describe().round(1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# chart 1 - status pie
status_counts = cont_df['status'].value_counts()
axes[0].pie(status_counts.values,
            labels=status_counts.index,
            colors=['#2E7D32','#C62828'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('SIP Continuity Status\n(Investors with 6+ SIPs)', fontweight='bold')

# chart 2 - avg gap histogram
axes[1].hist(cont_df['avg_gap_days'], bins=30,
             color='#1565C0', edgecolor='white', alpha=0.85)
axes[1].axvline(35, color='#C62828', linestyle='--', linewidth=2, label='35-day threshold')
axes[1].axvline(cont_df['avg_gap_days'].mean(), color='#2E7D32', linestyle='--',
                linewidth=2, label=f"Mean: {cont_df['avg_gap_days'].mean():.1f} days")
axes[1].set_xlabel('Avg Gap Between SIPs (days)')
axes[1].set_ylabel('Number of Investors')
axes[1].set_title('Distribution of SIP Gap Days', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS / 'sip_continuity_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: sip_continuity_chart.png")


## 7. Sector Concentration — Herfindahl-Hirschman Index (HHI)

`HHI = Σ(weight_i²)` where weight_i is each sector's % in the portfolio.

- HHI near 0 = well diversified across many sectors
- HHI near 10000 = fully concentrated in one sector
- Generally: HHI > 2500 = highly concentrated, < 1500 = diversified


In [ ]:
# compute sector weights per fund first
sector_weights = port_df.groupby(['amfi_code','sector'])['weight_pct'].sum().reset_index()

hhi_records = []

for code, group in sector_weights.groupby('amfi_code'):
    weights = group['weight_pct'].values
    # normalise to sum to 100 in case of rounding
    weights = weights / weights.sum() * 100
    hhi     = np.sum((weights) ** 2)
    top_sector = group.sort_values('weight_pct', ascending=False).iloc[0]['sector']
    top_weight = group['weight_pct'].max()

    hhi_records.append({
        'amfi_code'       : code,
        'scheme_name'     : fund_names.get(code, str(code)),
        'hhi_score'       : round(hhi, 1),
        'num_sectors'     : len(group),
        'top_sector'      : top_sector,
        'top_sector_wt'   : round(top_weight, 1),
        'concentration'   : 'High' if hhi > 2500 else ('Moderate' if hhi > 1500 else 'Diversified'),
    })

hhi_df = pd.DataFrame(hhi_records).sort_values('hhi_score', ascending=False).reset_index(drop=True)
hhi_df.to_csv(REPORTS / 'sector_hhi.csv', index=False)

print("sector_hhi.csv saved.")
print()
print("Top 10 most concentrated funds (by HHI):")
print(hhi_df[['scheme_name','hhi_score','num_sectors','top_sector','concentration']].head(10).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

colors_hhi = hhi_df['concentration'].map(
    {'High':'#C62828','Moderate':'#E65100','Diversified':'#2E7D32'}
)

bars = ax.barh(
    hhi_df['scheme_name'].str[:38][::-1],
    hhi_df['hhi_score'][::-1],
    color=colors_hhi[::-1], edgecolor='white', linewidth=0.4
)

ax.axvline(2500, color='#C62828', linestyle='--', linewidth=1.5, alpha=0.7, label='High (>2500)')
ax.axvline(1500, color='#E65100', linestyle='--', linewidth=1.5, alpha=0.7, label='Moderate (>1500)')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#C62828', label='High Concentration'),
    Patch(facecolor='#E65100', label='Moderate'),
    Patch(facecolor='#2E7D32', label='Diversified'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
ax.set_xlabel('HHI Score (higher = more concentrated)', fontsize=11)
ax.set_title('Sector Concentration (HHI) — All Funds\nHigher HHI = fewer sectors dominate the portfolio',
             fontsize=13, fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(REPORTS / 'sector_hhi_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: sector_hhi_chart.png")


## 8. Key Advanced Analytics Insights

Based on all analysis done in this notebook, here are the 5 most important findings:


In [ ]:
print("=" * 65)
print("ADVANCED ANALYTICS — KEY INSIGHTS")
print("=" * 65)

# Insight 1 — VaR
worst_var = var_df.iloc[0]
best_var  = var_df.iloc[-1]
print(f"\n1. RISK (VaR)")
print(f"   Highest risk fund : {worst_var['scheme_name'][:50]}")
print(f"   Daily VaR 95%     : {worst_var['var_95_pct']:.3f}% | CVaR: {worst_var['cvar_95_pct']:.3f}%")
print(f"   Lowest risk fund  : {best_var['scheme_name'][:50]}")
print(f"   Daily VaR 95%     : {best_var['var_95_pct']:.3f}%")

# Insight 2 — Cohort
best_cohort = cohort_stats.sort_values('avg_sip_amount', ascending=False).iloc[0]
print(f"\n2. INVESTOR COHORTS")
print(f"   Cohort {int(best_cohort['cohort_year'])} investors have the highest avg SIP: ₹{best_cohort['avg_sip_amount']:,}")
print(f"   Total cohorts analysed: {len(cohort_stats)}")
for _, row in cohort_stats.iterrows():
    print(f"   {int(row['cohort_year'])}: {int(row['num_investors'])} investors | Avg SIP ₹{int(row['avg_sip_amount']):,} | Total ₹{row['total_invested_cr']:.1f} Cr")

# Insight 3 — SIP continuity
print(f"\n3. SIP CONTINUITY")
print(f"   Investors analysed (6+ SIPs) : {len(cont_df):,}")
print(f"   Active rate                  : {(cont_df['status']=='Active').mean()*100:.1f}%")
print(f"   At-Risk rate                 : {(cont_df['status']=='At-Risk').mean()*100:.1f}%")
print(f"   Avg gap between SIPs         : {cont_df['avg_gap_days'].mean():.1f} days")

# Insight 4 — HHI
high_conc = hhi_df[hhi_df['concentration']=='High']
diversified = hhi_df[hhi_df['concentration']=='Diversified']
print(f"\n4. SECTOR CONCENTRATION (HHI)")
print(f"   Highly concentrated funds : {len(high_conc)}")
print(f"   Diversified funds         : {len(diversified)}")
print(f"   Most concentrated         : {hhi_df.iloc[0]['scheme_name'][:50]} (HHI={hhi_df.iloc[0]['hhi_score']})")
print(f"   Most diversified          : {hhi_df.iloc[-1]['scheme_name'][:50]} (HHI={hhi_df.iloc[-1]['hhi_score']})")

# Insight 5 — overall
print(f"\n5. OVERALL RISK SUMMARY")
avg_var = var_df['var_95_pct'].mean()
avg_cvar= var_df['cvar_95_pct'].mean()
print(f"   Industry avg VaR  (95%) : {avg_var:.3f}% per day")
print(f"   Industry avg CVaR (95%) : {avg_cvar:.3f}% per day")
print(f"   Annualised equiv VaR    : {avg_var * np.sqrt(252):.2f}%")

print(f"\nAll output files saved to: {REPORTS}")
